In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Cached-embedding MLP training can still run, but embedding extraction will be slow.")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
!rm -rf /content/music-genre-classification
!cp -r "/content/drive/MyDrive/music-genre-classification" /content/music-genre-classification
%cd /content/music-genre-classification

!pip install -q -r requirements.txt


In [ ]:
import subprocess
from pathlib import Path

embedding_caches = [
    Path("features/panns_cnn14_embeddings.npz"),
    Path("features/ast_embeddings.npz"),
]
missing_embeddings = [path for path in embedding_caches if not path.exists()]

if not missing_embeddings:
    print("Embedding caches ready:", ", ".join(str(path) for path in embedding_caches))
else:
    print("Missing embedding cache(s):", ", ".join(str(path) for path in missing_embeddings))
    if not Path("features/mel_specs.npz").exists():
        raise FileNotFoundError(
            "Missing features/mel_specs.npz. It is only required when an embedding "
            "cache must be rebuilt."
        )

    print("Preparing raw FMA audio so the missing embeddings can be extracted.")
    data_dir = Path("data")
    data_dir.mkdir(exist_ok=True)
    audio_root = data_dir / "fma_small"
    archive_path = data_dir / "fma_small.zip"
    if not audio_root.exists():
        if not archive_path.exists():
            subprocess.run(
                [
                    "wget",
                    "-c",
                    "-O",
                    str(archive_path),
                    "https://os.unil.cloud.switch.ch/fma/fma_small.zip",
                ],
                check=True,
            )
        subprocess.run(["unzip", "-q", "-n", str(archive_path), "-d", str(data_dir)], check=True)


In [ ]:
!python3 transfer_learning_panns_cnn14.py


In [ ]:
!python3 transfer_learning_ast.py


In [ ]:
import pandas as pd
from pathlib import Path

paths = [
    Path("results/5.2 PANNs-CNN14/metrics.csv"),
    Path("results/5.3 AST/metrics.csv"),
]
frames = []
for path in paths:
    if path.exists():
        df = pd.read_csv(path)
        df["experiment"] = path.parent.name
        frames.append(df)

summary = pd.concat(frames, ignore_index=True)
summary[summary["split"] == "test"].sort_values("f1_macro", ascending=False)


In [ ]:
from pathlib import Path
import shutil

drive_root = Path("/content/drive/MyDrive/music-genre-classification")
drive_results = drive_root / "results"
drive_features = drive_root / "features"
drive_results.mkdir(parents=True, exist_ok=True)
drive_features.mkdir(parents=True, exist_ok=True)

for name in ["5.2 PANNs-CNN14", "5.3 AST"]:
    src = Path("results") / name
    dst = drive_results / name
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print("Copied", src, "->", dst)

for filename in ["model_comparison.csv", "model_comparison.png"]:
    src = Path("results") / filename
    if src.exists():
        shutil.copy2(src, drive_results / filename)
        print("Copied", src, "->", drive_results / filename)

for filename in ["panns_cnn14_embeddings.npz", "ast_embeddings.npz"]:
    src = Path("features") / filename
    if src.exists():
        shutil.copy2(src, drive_features / filename)
        print("Copied", src, "->", drive_features / filename)
